# Taller en parejas — De Kaggle a dos modelos de regresión

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 2

Hoy ustedes hacen el pipeline completo, de punta a punta, con un dataset que eligen en Kaggle: **datos → preprocesamiento → modelo → evaluación → interpretación**. Es un ensayo de la **Entrega 2** del proyecto, con otro dataset y en pareja.

**La pregunta es de regresión** (la *y* es un número). Así pueden usar los dos modelos que ya vieron —**regresión lineal** y **árbol de decisión**— y compararlos contra la **línea base**.

> 💾 **Antes de escribir nada:** *Archivo → Guardar una copia en Drive* y compartan esa copia entre los dos (botón *Compartir*, con permiso de edición).

## Cómo trabajan
- **Piloto y copiloto.** Uno escribe, el otro dicta, revisa y busca en la documentación. **Cambian cada 20 minutos** (el profesor avisa). Si uno programa más que el otro, el que programa menos empieza de piloto.
- **Cada gráfica y cada métrica lleva una frase** en markdown: *qué muestra* y *qué concluyen*. Una gráfica sin frase no cuenta.
- **Pueden copiar código de las sesiones** (la 2, la 3 y la 4 tienen casi todo lo que necesitan). Lo que no pueden copiar es la conclusión.
- Las **pistas** están plegadas debajo de cada sección: ábranlas solo si se atascan más de 5 minutos.

## Qué se entrega
Este notebook, ejecutado de arriba abajo sin errores, con el nombre `taller_<apellido1>_<apellido2>.ipynb`, por el canal del curso, **antes de la próxima clase**.

| Criterio | 1 · No está | 2 · A medias | 3 · Bien | 4 · Muy bien |
|---|---|---|---|---|
| **Partición sin fuga** | no hay partición, o se preprocesa con todos los datos | se parte, pero algo se ajusta con prueba | se parte antes y todo se ajusta solo con entrenamiento | además revisan columnas sospechosas y explican por qué quitaron (o dejaron) cada una |
| **Contra la línea base** | no hay línea base | está, pero no se compara | los dos modelos se comparan contra `DummyRegressor` en prueba | además explican cuánto mejora cada uno, en las unidades de la *y* |
| **Métricas bien leídas** | solo números | MAE, RMSE y R² sin unidades ni lectura | con unidades y una frase cada una | además revisan los residuos (supuestos) y dicen qué le falta al modelo |
| **Conclusión honesta** | no hay | "el mejor es X" sin razones | eligen modelo con argumentos | además dicen qué **no** pueden concluir y qué harían con más tiempo |


## Los datasets propuestos

Elijan **uno** con su pareja. Antes de cargarlo, **ábranlo en Kaggle** y lean la ficha: de dónde salen los datos, qué significa cada columna y qué licencia tienen (eso va en la sección 1).

| # | Dataset | En Kaggle | La *y* | Tamaño | Para quién |
|---|---|---|---|---|---|
| 1 | **Diamantes** (precio de 53 940 diamantes según quilates, corte, color y pureza) | [shivam2503/diamonds](https://www.kaggle.com/datasets/shivam2503/diamonds) | `price` (USD) | grande, limpio | para empezar |
| 2 | **Vivienda en California** (censo de 1990, una fila por zona) | [camnugent/california-housing-prices](https://www.kaggle.com/datasets/camnugent/california-housing-prices) | `median_house_value` (USD) | 20 640 × 10 | intermedio |
| 3 | **Bicicletas compartidas** (alquileres por hora en Washington D. C., 2011–2012) | buscar *bike sharing dataset* (el original está en [UCI](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)) | `cnt` (bicicletas alquiladas en la hora) | 17 379 × 17 | intermedio |
| 4 | **Estudiantes de secundaria en Portugal** (notas de portugués y vida familiar) | [larsen0966/student-performance-data-set](https://www.kaggle.com/datasets/larsen0966/student-performance-data-set) | `G3` (nota final, 0 a 20) | 649 × 33 | para quien quiera pensar |
| 5 | **Otro de Kaggle, elegido por ustedes** | — | numérica | ≥ 1 000 filas | para quien ya tenga uno en mente |

**Si eligen la opción 5:** la *y* tiene que ser numérica, sin texto libre como variable principal y con al menos 1 000 filas. Descarguen el CSV de Kaggle (botón *Download*) y súbanlo a Colab con la celda de la sección 0. Si es el mismo dataset de su proyecto individual, no pasa nada: este trabajo les adelanta la Entrega 2.

Cada dataset tiene **algo escondido** que, si no lo ven, les da un resultado que parece muy bueno y no lo es. Encontrarlo es parte del taller.

## 0. Setup — elijan el dataset y cárguenlo

Para no depender de Kaggle (pide cuenta y a veces no responde), el curso tiene una **copia** de los cuatro datasets propuestos en su repositorio. Cambien `ELEGIDO` por el número de su dataset y ejecuten.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import requests

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"

ELEGIDO = 1          # 1 diamantes · 2 vivienda · 3 bicicletas · 4 estudiantes · 5 el suyo

DATASETS = {
    1: ("taller_diamantes.csv.gz", "price"),
    2: ("taller_vivienda_california.csv.gz", "median_house_value"),
    3: ("taller_bicicletas_hora.csv.gz", "cnt"),
    4: ("taller_estudiantes_portugues.csv.gz", "G3"),
}

def bajar(archivo, ruta):
    """Baja un archivo del repositorio del curso si aún no está en el disco de Colab."""
    if Path(archivo).exists():
        return
    r = requests.get(REPO + ruta, timeout=120)
    r.raise_for_status()
    Path(archivo).write_bytes(r.content)

bajar("figuras.py", "tools/figuras.py")
import figuras as fg

if ELEGIDO in DATASETS:
    archivo, Y = DATASETS[ELEGIDO]
    bajar(archivo, "datos/" + archivo)
    df = pd.read_csv(archivo)
else:
    from google.colab import files              # opción 5: suban el CSV que bajaron de Kaggle
    subido = files.upload()
    archivo = next(iter(subido))
    df = pd.read_csv(archivo)
    Y = "ESCRIBAN_AQUI_EL_NOMBRE_DE_LA_Y"
print(archivo, df.shape, "· y =", Y)

## 1. La pregunta (5 min)

Respondan en esta celda, en sus palabras:

- **¿Qué quieren predecir y para quién sería útil?** (ej.: *"el precio de un diamante, para que un joyero sepa si una oferta es justa"*).
- **¿De dónde salen los datos?** (quién los recogió, cuándo, licencia: está en la ficha de Kaggle).
- **¿Qué fila es una observación?** (un diamante, una zona, una hora, un estudiante).
- **Regla de la *y*:** ¿en qué momento se conoce la *y*? ¿Qué columnas se conocen **antes** que ella?

*Escriban aquí.*

## 2. Primera mirada (10 min)

Tamaño, tipos, vacíos y duplicados: lo de la Sesión 2. Al final, escriban qué encontraron y qué van a hacer con cada cosa.

<details><summary>Pista</summary>

```python
df.info()
df.isna().mean().mul(100).round(1).sort_values(ascending=False).head(10)   # % de vacíos
df.duplicated().sum()
df.describe().T
```
</details>

In [ ]:
# Escriban aquí

**Lo que encontramos y qué haremos:** *escriban aquí (vacíos, duplicados, tipos raros, valores imposibles).*

## 3. Exploración (EDA) y estadística descriptiva (20 min)

Mínimo **tres gráficas**, cada una con su frase:

1. La **distribución de la *y*** (histograma). ¿Es simétrica? ¿Tiene valores extremos o un tope?
2. La *y* contra la **variable numérica que crean más importante** (dispersión).
3. La *y* por una **variable categórica** (cajas o barras de promedios).

Y una **tabla de correlaciones** de las numéricas con la *y*, ordenada. 👀 **Antes de calcularla, escriban cuál creen que sale primera.**

<details><summary>Pista</summary>

```python
df[Y].hist(bins=50)
df.plot.scatter(x="NOMBRE_DE_UNA_X", y=Y, alpha=0.3, s=5)
df.boxplot(column=Y, by="NOMBRE_DE_UNA_CATEGORICA", rot=45)
df.select_dtypes("number").corr()[Y].sort_values(ascending=False)
```
</details>

In [ ]:
# Escriban aquí

In [ ]:
# Escriban aquí

In [ ]:
# Escriban aquí

**Qué muestran y qué concluimos** (una frase por gráfica): *escriban aquí.*

## 4. ¿Hay columnas sospechosas? (5 min)

Para cada columna, la pregunta de la Sesión 3: **¿la tendría en la mano antes de conocer la *y*?** Revisen sobre todo las que en la tabla de correlaciones salieron **demasiado** altas.

| Columna | ¿Se conoce antes que la *y*? | ¿La dejamos? ¿Por qué? |
|---|---|---|
| | | |
| | | |

## 5. Partición y preprocesamiento (15 min)

En este orden, que importa:

1. Quiten las columnas que decidieron quitar (sección 4) y las que no sirven como X (identificadores, fechas en texto).
2. **Partan primero**: `train_test_split(..., test_size=0.2, random_state=42)`.
3. Después preprocesen, **ajustando todo solo con entrenamiento**: vacíos (con la mediana de entrenamiento), *one-hot* con `drop_first=True` y, si van a usar Ridge, Lasso o k-NN, escalar.

<details><summary>Pista</summary>

```python
from sklearn.model_selection import train_test_split
X = df.drop(columns=[Y])                       # + las columnas que decidieron quitar
X = pd.get_dummies(X, drop_first=True, dtype=int)
y = df[Y]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
medianas = X_tr.median()                        # SOLO de entrenamiento
X_tr, X_te = X_tr.fillna(medianas), X_te.fillna(medianas)
print(X_tr.shape, X_te.shape)
```
(Hacer el *one-hot* antes de partir no filtra información de la *y*: solo crea columnas. Lo que no se puede es calcular medianas, promedios o escalas con prueba.)
</details>

In [ ]:
# Escriban aquí

## 6. Línea base y dos modelos (20 min)

🔍 **Antes de entrenar, escriban su predicción:** ¿cuál creen que gana, la regresión lineal o el árbol? ¿Por cuánto le ganará a la línea base?

**Su predicción:** ____

Entrenen:
- `DummyRegressor()` — la línea base: predice siempre el promedio.
- `LinearRegression()` — Sesión 4.
- `DecisionTreeRegressor(max_depth=..., random_state=42)` — el árbol de las Sesiones 1–3, ahora para un número: en cada hoja predice el **promedio** de la *y* de esa hoja. Prueben 2 o 3 profundidades y quédense con una.

**¿Quieren probar un modelo que no hemos visto?** Adelante (`RandomForestRegressor`, `KNeighborsRegressor`, `Ridge`, `Lasso`…), con dos condiciones: también se compara contra la línea base, y escriben en tres líneas su ficha **Dentro del `.fit()`**: qué minimiza, moviendo qué y cómo lo busca.

<details><summary>Pista</summary>

```python
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

modelos = {
    "línea base": DummyRegressor(),
    "regresión lineal": LinearRegression(),
    "árbol (prof. 6)": DecisionTreeRegressor(max_depth=6, random_state=42),
}
for nombre, m in modelos.items():
    m.fit(X_tr, y_tr)
```
</details>

In [ ]:
# Escriban aquí

## 7. Evaluación: métricas y residuos (15 min)

La función `tabla_metricas` ya está hecha: arma la tabla con **MAE, RMSE y R²** en **prueba** para todos sus modelos. Después, el diagnóstico de supuestos de la Sesión 4 para la regresión lineal.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def tabla_metricas(modelos, X, y):
    """MAE, RMSE y R² de cada modelo (ya entrenado) sobre X, y. Ordena del mejor al peor R²."""
    filas = {}
    for nombre, m in modelos.items():
        p = m.predict(X)
        filas[nombre] = {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** 0.5, "R²": r2_score(y, p)}
    return pd.DataFrame(filas).T.sort_values("R²", ascending=False).round(3)

# tabla_metricas(modelos, X_te, y_te)

In [ ]:
# 1) la tabla de métricas en prueba
# 2) fg.diagnostico_supuestos(y_te, modelos["regresión lineal"].predict(X_te));
# 3) fg.vif(X_tr).head(10)

**Lectura de las métricas** (con unidades): *"el árbol se equivoca en promedio por ___ (unidades de la y), contra ___ de la línea base…"*. *Escriban aquí.*

**Lectura de los residuos:** ¿qué supuestos fallan? ¿Qué le falta al modelo? *Escriban aquí.*

**¿Algún resultado es demasiado bueno?** Un R² por encima de 0,95 casi nunca es buena noticia: vuelvan a la sección 4. *Escriban aquí.*

## 8. Conclusión (10 min)

Cinco frases, sin código:

1. **Qué modelo eligen y por qué** (no solo "el de mayor R²": ¿cuál le explicarían a quien va a usarlo?).
2. **Cuánto mejora sobre la línea base**, en las unidades de la *y*.
3. **Qué columna o decisión cambió más el resultado.**
4. **Qué NO pueden concluir** con este modelo (¿causa? ¿otras ciudades, otros años?).
5. **Qué harían con una semana más.**

*Escriban aquí.*

---
**Lista de chequeo antes de entregar:** ☐ *Entorno de ejecución → Reiniciar y ejecutar todo* corre sin errores · ☐ cada gráfica y cada métrica tiene su frase · ☐ la partición está antes de cualquier `.fit()` · ☐ hay línea base · ☐ el nombre del archivo tiene los dos apellidos.